# Aula 10 — Embeddings contextuais e Transformers

**Quando a representação de uma palavra depende da frase em que ela aparece**

Na Aula 9, vimos que Word2Vec e FastText atribuem essencialmente uma representação fixa a cada palavra.

Agora vamos enfrentar uma limitação fundamental:

```text
banco aprovou o crédito
sentei no banco da praça
```

A palavra `banco` aparece nas duas frases, mas com sentidos diferentes.

Embeddings contextuais foram criados justamente para permitir que a representação de um token dependa das palavras ao redor dele.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar a diferença entre embeddings estáticos e contextuais;
- compreender por que contexto altera representação;
- explicar, em alto nível, o papel do mecanismo de atenção;
- reconhecer a arquitetura Transformer como base de modelos modernos de linguagem;
- gerar embeddings contextuais com um modelo pré-treinado;
- comparar representações da mesma palavra em frases diferentes;
- interpretar similaridade contextual com cuidado.


## 📘 Glossário da aula

Conceitos centrais: **embedding contextual · Transformer · atenção · self-attention · token contextual · modelo pré-treinado · encoder**.

- [Glossário PT-BR](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md)
- [Glossary EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.


## 2. O problema da polissemia

Palavras podem assumir significados diferentes conforme o contexto.

Exemplo:

```text
manga da camisa
manga madura
```

Um embedding estático tende a usar a mesma representação-base para `manga` nos dois casos.

Um embedding contextual produz representações diferentes porque considera as outras palavras da frase.


## 3. A ideia de atenção

O mecanismo de atenção permite que cada token considere outros tokens da sequência ao construir sua representação.

Em termos intuitivos:

```text
token atual
→ observa outros tokens
→ atribui pesos de relevância
→ combina informação contextual
→ gera nova representação
```

No caso de **self-attention**, os tokens de uma mesma sequência interagem entre si.


## 4. Transformer em alto nível

Transformers organizam várias operações de atenção e transformação vetorial em camadas.

A grande mudança foi permitir processamento altamente paralelo e contextualização profunda de sequências.

Para esta aula, retenha esta ideia:

> um Transformer recebe tokens e devolve representações contextualizadas de cada posição da sequência.


## 5. Preparando um experimento com um modelo pré-treinado

Vamos usar um modelo pequeno da biblioteca `transformers` apenas para observar embeddings contextuais.

Como o notebook oficial do TIL mantém internet desabilitada, este exemplo depende de o modelo estar disponível no ambiente Kaggle ou ser anexado como recurso. Por isso, a célula abaixo foi escrita para falhar de forma informativa caso o modelo não esteja disponível localmente.


### Arquitetura preferida do TIL para modelos pré-treinados

O fluxo oficial preferido para esta aula é usar um **Kaggle Model anexado e versionado**, permitindo que o notebook execute com a dependência explícita e, idealmente, com internet desabilitada.

```text
Kaggle Model versionado
→ anexado ao notebook
→ modelo disponível localmente na execução
→ Internet OFF
```

Enquanto o recurso oficial desta aula não estiver anexado e validado, o código permanece tolerante à ausência do modelo.

> Não usamos caminhos Kaggle inventados nem handles não validados. O identificador exato do recurso será registrado somente após validação real no Kaggle.


In [ ]:
MODEL_NAME = "distilbert-base-multilingual-cased"
MODEL_DIR = "/kaggle/input/models/goddiao/distilbert-base-multilingual-cased/pytorch/default/1/distilbert-base-multilingual-cased"

print("Modelo planejado:", MODEL_NAME)
print("Objetivo: observar representações contextuais, não fazer fine-tuning.")


## 6. Tokenização subword

Transformers modernos normalmente não trabalham apenas com palavras inteiras. Eles usam tokenizadores capazes de dividir palavras em subpalavras.

Isso reduz problemas com palavras raras e permite reutilizar partes recorrentes.

Exemplo conceitual:

```text
inacreditável
→ ina + credit + ável   (exemplo apenas ilustrativo)
```

A divisão real depende do vocabulário do tokenizador.


## 7. Carregando tokenizer e encoder

A próxima célula tenta carregar tokenizer e modelo apenas de arquivos locais.


In [ ]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModel

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_DIR,
        local_files_only=True,
    )

    encoder = AutoModel.from_pretrained(
        MODEL_DIR,
        local_files_only=True,
    )

    encoder.eval()
    MODEL_AVAILABLE = True

    print("Modelo Kaggle carregado localmente.")
    print("Tokenizer:", type(tokenizer).__name__)
    print("Encoder:", type(encoder).__name__)
    print("Model type:", encoder.config.model_type)
    print("Vocab size:", encoder.config.vocab_size)

except Exception as exc:
    MODEL_AVAILABLE = False
    print("Modelo não disponível localmente neste ambiente.")
    print("Motivo resumido:", type(exc).__name__)
    print("Detalhe:", exc)


### Por que tratar essa possibilidade explicitamente?

Porque reprodutibilidade inclui reconhecer dependências externas.

Um notebook não deve fingir que um artefato externo está disponível quando não está.

Mais adiante poderemos anexar modelos ao Kaggle como recurso oficial do curso.


## 8. Comparando a mesma palavra em dois contextos

Vamos usar duas frases em inglês para combinar com o modelo escolhido:

```text
the bank approved the loan
we sat by the river bank
```

A palavra `bank` assume sentidos diferentes.


In [ ]:
sentences = [
    "the bank approved the loan",
    "we sat by the river bank",
]

sentences


## 9. Extraindo embeddings contextuais

Se o modelo estiver disponível, a função abaixo extrai a representação do token `bank` em cada frase.


In [ ]:
def contextual_embedding(sentence, target_token):
    if not MODEL_AVAILABLE:
        raise RuntimeError("Modelo contextual não está disponível localmente.")

    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = encoder(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    positions = [i for i, token in enumerate(tokens) if token == target_token]

    if not positions:
        raise ValueError(f"Token {target_token!r} não encontrado após tokenização.")

    position = positions[0]
    vector = outputs.last_hidden_state[0, position].numpy()
    return tokens, vector


In [ ]:
if MODEL_AVAILABLE:
    tokens_1, bank_finance = contextual_embedding(sentences[0], "bank")
    tokens_2, bank_river = contextual_embedding(sentences[1], "bank")

    print(tokens_1)
    print(tokens_2)
    print("Dimensão:", len(bank_finance))
else:
    print("Experimento contextual não executado porque o modelo não está disponível localmente.")


## 10. Comparando as duas representações

Agora calculamos similaridade por cosseno entre as duas ocorrências de `bank`.


In [ ]:
if MODEL_AVAILABLE:
    import numpy as np

    similarity = np.dot(bank_finance, bank_river) / (
        np.linalg.norm(bank_finance) * np.linalg.norm(bank_river)
    )
    print("Similaridade contextual:", round(float(similarity), 3))
else:
    print("Sem cálculo: modelo não disponível localmente.")


### O que interpretar

As duas ocorrências vêm do mesmo token lexical, mas o Transformer produz vetores diferentes porque o contexto é diferente.

Isso é justamente o que embeddings estáticos não conseguem fazer diretamente.

**Checkpoint:** contexto passou a fazer parte da representação.


## 11. Atenção não é explicação perfeita

É tentador interpretar pesos de atenção como uma explicação completa do raciocínio do modelo.

Essa conclusão é forte demais.

Pesos de atenção podem oferecer sinais úteis sobre interação entre tokens, mas não devem ser tratados automaticamente como uma explicação causal ou total da decisão do modelo.


## 12. Do encoder aos modelos de linguagem modernos

Modelos como BERT popularizaram encoders contextuais profundos.

Outras famílias de Transformers evoluíram para geração autoregressiva de texto e deram origem aos grandes modelos de linguagem modernos.

A base conceitual que estamos construindo é:

```text
tokens
→ embeddings
→ atenção
→ representação contextual
→ tarefa downstream
```


## 13. Exercício guiado

Considere as frases:

```python
exercise_sentences = [
    'the bat flew at night',
    'the player swung the bat',
]
```

Seu código deve, **se o modelo estiver disponível localmente**:

1. extrair o embedding contextual de `bat` nas duas frases;
2. calcular a similaridade por cosseno;
3. imprimir a similaridade;
4. explicar em uma frase por que os vetores não precisam ser idênticos.


In [ ]:
# Escreva sua solução aqui.

exercise_sentences = [
    'the bat flew at night',
    'the player swung the bat',
]

# Continue a partir daqui.


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q10.hint()` e `q10.solution()`.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q10 = TILExercise(
    hint_text=(
        "Reutilize a função `contextual_embedding()`. "
        "Depois calcule cosseno com `np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))`. "
        "Proteja a execução com `if MODEL_AVAILABLE:`."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "if MODEL_AVAILABLE:\n"
        "    _, bat_animal = contextual_embedding(exercise_sentences[0], 'bat')\n"
        "    _, bat_object = contextual_embedding(exercise_sentences[1], 'bat')\n\n"
        "    similarity = np.dot(bat_animal, bat_object) / (\n"
        "        np.linalg.norm(bat_animal) * np.linalg.norm(bat_object)\n"
        "    )\n"
        "    print('Similaridade:', round(float(similarity), 3))\n"
        "    print('Os vetores diferem porque o contexto de bat é diferente nas duas frases.')\n"
        "else:\n"
        "    print('Modelo não disponível localmente.')\n"
        "```"
    ),
)

print("Exercício preparado. Tente resolver antes de usar q10.hint() ou q10.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q10.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q10.solution()


## 14. Reprodutibilidade

- linguagem: Python;
- bibliotecas: `numpy`, `torch`, `transformers`;
- modelo planejado: `distilbert-base-uncased`;
- uso: inferência apenas;
- internet: desabilitada;
- carregamento: `local_files_only=True`;
- acelerador: CPU;
- comportamento esperado: experimento executa apenas se o modelo estiver disponível localmente.


## 15. Resumo

Nesta aula, você aprendeu que:

- embeddings estáticos não representam diretamente sentidos diferentes da mesma palavra;
- embeddings contextuais dependem da frase;
- self-attention permite interação entre tokens de uma mesma sequência;
- Transformers produzem representações contextualizadas em várias camadas;
- tokenização subword ajuda a lidar com vocabulários grandes e palavras raras;
- atenção não deve ser interpretada automaticamente como explicação completa;
- embeddings contextuais são uma ponte fundamental para modelos de linguagem modernos.

### Ideia principal

```text
A palavra é a mesma.
O contexto muda.
A representação também pode mudar.
```

**Fim da Aula 10.**
